<a href="https://colab.research.google.com/github/lunecarvalho/newslens-development/blob/main/newslens_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd

In [2]:
from google.colab import files

uploaded = files.upload()

Saving initial_dataset.zip to initial_dataset.zip


In [3]:
import zipfile

zip_path = '/content/initial_dataset.zip'

extract_path = '/content/initial_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraído com sucesso!")

Dataset extraído com sucesso!


In [4]:
for raiz, pastas, arquivos in os.walk(extract_path):
    print(raiz, "->", len(arquivos), "arquivos")

/content/initial_dataset -> 0 arquivos
/content/initial_dataset/initial_dataset -> 0 arquivos
/content/initial_dataset/initial_dataset/fake -> 3600 arquivos
/content/initial_dataset/initial_dataset/true -> 3600 arquivos


In [5]:
fake_path = None
true_path = None

for raiz, pastas, arquivos in os.walk(extract_path):
    if os.path.basename(raiz).lower() == 'fake':
        fake_path = raiz

    if os.path.basename(raiz).lower() == 'true':
        true_path = raiz

print("Fake:", fake_path)
print("True:", true_path)

Fake: /content/initial_dataset/initial_dataset/fake
True: /content/initial_dataset/initial_dataset/true


In [6]:
dados = []

In [7]:
for arquivo in os.listdir(fake_path):

    if arquivo.lower().endswith('.txt'):

        caminho = os.path.join(fake_path, arquivo)

        with open(caminho, 'r', encoding='utf-8') as f:
            texto = f.read()

        dados.append({
            'label': 'falso',
            'arquivo': arquivo,
            'texto': texto,
            'tema': ''
        })

In [8]:
for arquivo in os.listdir(true_path):

    if arquivo.lower().endswith('.txt'):

        caminho = os.path.join(true_path, arquivo)

        with open(caminho, 'r', encoding='utf-8') as f:
            texto = f.read()

        dados.append({
            'label': 'verdadeiro',
            'arquivo': arquivo,
            'texto': texto,
            'tema': ''
        })

In [9]:
df = pd.DataFrame(dados)

In [10]:
df.insert(0, 'id', range(1, len(df) + 1))

In [11]:
df.head()

,id,label,arquivo,texto,tema
0,1,falso,1170.txt,Falso médico é preso pela PM de Santa Catarina...,
1,2,falso,1688.txt,Antes da confirmação da morte de Teori Zavasck...,
2,3,falso,1863.txt,Senador que assume o lugar de Renan foi flagra...,
3,4,falso,1563.txt,"Repórter relata conversa entre deputados: ""A D...",
4,5,falso,3556.txt,Governo repressor e comunista usa exército par...,


In [12]:
df.to_csv(
    '/content/initial_dataset.csv',
    index=False,
    encoding='utf-8-sig'
)

print('CSV criado com sucesso!')

CSV criado com sucesso!


In [13]:
print(df.columns.tolist())

['id', 'label', 'arquivo', 'texto', 'tema']


In [14]:
print(f'Total de textos: {len(df)}')
print(df['label'].value_counts())

Total de textos: 7200
label
falso         3600
verdadeiro    3600
Name: count, dtype: int64


In [15]:
files.download('/content/initial_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print('Linhas duplicadas:', df.duplicated().sum())

print('Textos duplicados:', df['texto'].duplicated().sum())

print('Textos com labels diferentes:',
      (df.groupby('texto')['label'].nunique() > 1).sum())

Linhas duplicadas: 0
Textos duplicados: 1
Textos com labels diferentes: 0


In [17]:
duplicados = df[df['texto'].duplicated(keep=False)].sort_values('texto')

duplicados[['id', 'label', 'arquivo', 'texto']]

,id,label,arquivo,texto
4077,4078,verdadeiro,69.txt,Suplicy participará de programa de Doria na we...
5545,5546,verdadeiro,61.txt,Suplicy participará de programa de Doria na we...


In [18]:
print(duplicados.iloc[0]['texto'])
print('---')
print(duplicados.iloc[1]['texto'])

Suplicy participará de programa de Doria na web nesta quinta

O vereador Eduardo Suplicy (PT-SP) participará nesta quinta-feira (10), às 20h30, do programa "Olho no Olho", transmitido pelas redes sociais do prefeito João Doria (PSDB-SP).

Doria já recebeu no quadro aliados e personalidades como o cantor Lobão, o apresentador José Luiz Datena, o ex-jogador de basquete Oscar, o cantor Roger, do Ultraje a Rigor, e a jornalista Joice Hasselman.

No começo de sua gestão, o tucano poupou de críticas seu antecessor Fernando Haddad (PT-SP), mas, nos últimos meses, passou a acusar o ex-prefeito de deixar um rombo de R$ 7 bilhões na prefeitura. Haddad nega e diz que deixou as contas da cidade em ordem.

Suplicy, por sua vez, é uma das vozes críticas à Doria na Câmara dos Vereadores. 

---
Suplicy participará de programa de Doria na web nesta quinta

O vereador Eduardo Suplicy (PT-SP) participará nesta quinta-feira (10), às 20h30, do programa "Olho no Olho", transmitido pelas redes sociais do pre

In [19]:
df = df.drop_duplicates(subset='texto', keep='first').reset_index(drop=True)

In [20]:
df['id'] = range(1, len(df) + 1)

In [21]:
print('Total de textos:', len(df))
print('Textos duplicados:', df['texto'].duplicated().sum())

Total de textos: 7199
Textos duplicados: 0


In [22]:
df.to_csv(
    '/content/initial_dataset.csv',
    index=False,
    encoding='utf-8-sig'
)

print('initial_dataset.csv atualizado com sucesso!')

initial_dataset.csv atualizado com sucesso!


In [23]:
df = df.rename(columns={'texto': 'texto_original'})

In [24]:
df['texto_preprocessado'] = df['texto_original']

In [25]:
print(df.columns.tolist())

['id', 'label', 'arquivo', 'texto_original', 'tema', 'texto_preprocessado']


In [26]:
df = df[
    ['id', 'label', 'arquivo', 'texto_original', 'texto_preprocessado', 'tema']
]

In [27]:
df.head()

,id,label,arquivo,texto_original,texto_preprocessado,tema
0,1,falso,1170.txt,Falso médico é preso pela PM de Santa Catarina...,Falso médico é preso pela PM de Santa Catarina...,
1,2,falso,1688.txt,Antes da confirmação da morte de Teori Zavasck...,Antes da confirmação da morte de Teori Zavasck...,
2,3,falso,1863.txt,Senador que assume o lugar de Renan foi flagra...,Senador que assume o lugar de Renan foi flagra...,
3,4,falso,1563.txt,"Repórter relata conversa entre deputados: ""A D...","Repórter relata conversa entre deputados: ""A D...",
4,5,falso,3556.txt,Governo repressor e comunista usa exército par...,Governo repressor e comunista usa exército par...,


In [28]:
df.to_csv(
    '/content/initial_dataset.csv',
    index=False,
    encoding='utf-8-sig'
)

print('initial_dataset.csv atualizado com sucesso!')

initial_dataset.csv atualizado com sucesso!


In [29]:
!pip install beautifulsoup4

In [48]:
import re
from bs4 import BeautifulSoup

In [49]:
def preprocessar_texto(texto):

    # Remover HTML
    texto = BeautifulSoup(texto, 'html.parser').get_text(separator=' ')

    # Remover URLs
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)

    # Remover caracteres estranhos
    texto = re.sub(
        r'[^\w\s.,!?;:()\-áàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ]',
        '',
        texto
    )

    # Normalizar espaços
    texto = re.sub(r'\s+', ' ', texto).strip()

    # Converter para minúsculas
    texto = texto.lower()

    return texto

In [50]:
df['texto_preprocessado'] = df['texto_original'].apply(preprocessar_texto)

In [51]:
print("TEXTO ORIGINAL:")
print(df.loc[57, 'texto_original'])

print("\n" + "="*80 + "\n")

print("TEXTO PREPROCESSADO:")
print(df.loc[57, 'texto_preprocessado'])

TEXTO ORIGINAL:
O desrespeito de Lula com o MP,  com a PF e com o Judiciário: "Chamem a mãe deles para depor". .  Antes que comecem o mi-mi-mi petista,  o vídeo foi editado e contém somente alguns trechos do discurso *** chato e repetitivo *** da Alma Mais Honesta Desse País Como pode alguém dizer que respeita (entre aspas) as instituições do país e depois vomitar ataques inconsequentes no microfone? Lula não respeita nada nem ninguém. Ele se acha acima das leis e usa um simples microfone como se fosse uma arma.



TEXTO PREPROCESSADO:
o desrespeito de lula com o mp, com a pf e com o judiciário: chamem a mãe deles para depor. . antes que comecem o mi-mi-mi petista, o vídeo foi editado e contém somente alguns trechos do discurso chato e repetitivo da alma mais honesta desse país como pode alguém dizer que respeita (entre aspas) as instituições do país e depois vomitar ataques inconsequentes no microfone? lula não respeita nada nem ninguém. ele se acha acima das leis e usa um simples mic

In [52]:
df.to_csv(
    '/content/initial_dataset.csv',
    index=False,
    encoding='utf-8-sig'
)

print('initial_dataset.csv atualizado com sucesso!')

initial_dataset.csv atualizado com sucesso!


In [55]:
print(df.loc[0, 'texto_preprocessado'])

falso médico é preso pela pm de santa catarina: sou formado em medicina pelo seriado greys anatomy. josias de farias júnior se passava por médico em um hospital da unimed em balneário camboriú, santa catarina. o estelionatário de 19 anos foi preso pela polícia na noite da última terça-feira (30). ele portava prontuários, carimbos médicos, um jaleco e um estetoscópio que, de acordo com testemunhas, foram roubados de um médico do próprio hospital. seguranças do local desconfiaram da credencial do jovem e o detiveram até a chegada da polícia. ainda não se sabe se ele chegou a atender algum paciente ou até mesmo prescreveu alguma receita. durante o depoimento, ele afirmou ser fã de um seriado de tv (sobre medicina) muito famoso nos eua: greys anatomy. em um vídeo postado nas redes sociais ( minuto 0:52 ) josias declara ser um fã apaixonado das séries greys anatomy e bones.


In [56]:
df = pd.read_csv('/content/initial_dataset.csv')
df.head()

,id,label,arquivo,texto_original,texto_preprocessado,tema
0,1,falso,1170.txt,Falso médico é preso pela PM de Santa Catarina...,falso médico é preso pela pm de santa catarina...,NaN
1,2,falso,1688.txt,Antes da confirmação da morte de Teori Zavasck...,antes da confirmação da morte de teori zavasck...,NaN
2,3,falso,1863.txt,Senador que assume o lugar de Renan foi flagra...,senador que assume o lugar de renan foi flagra...,NaN
3,4,falso,1563.txt,"Repórter relata conversa entre deputados: ""A D...",repórter relata conversa entre deputados: a di...,NaN
4,5,falso,3556.txt,Governo repressor e comunista usa exército par...,governo repressor e comunista usa exército par...,NaN
